# Grietas y riesgo — Entrenamiento MobileNetV2 en Colab

**Antes de ejecutar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

Necesitas en tu Google Drive una carpeta `grietas-riesgo` con dos archivos:
- `grietas-riesgo.zip` — el proyecto (carpeta `src/` con el `train_transfer.py` corregido)
- `archive.zip` — el dataset *Surface Crack Detection* descargado de Kaggle (contiene `Positive/` y `Negative/`)

Ejecuta las celdas en orden (Shift+Enter). Tiempo total aproximado: 10–15 minutos.

In [ ]:
# 1. Verificar que hay GPU
!nvidia-smi -L

In [ ]:
# 2. Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Rutas de los archivos en Drive (ajusta si los pusiste en otra carpeta)
PROYECTO_ZIP = '/content/drive/MyDrive/grietas-riesgo/grietas-riesgo.zip'
DATASET_ZIP  = '/content/drive/MyDrive/grietas-riesgo/archive.zip'
SALIDA_DRIVE = '/content/drive/MyDrive/grietas-riesgo/models'

import os
assert os.path.exists(PROYECTO_ZIP), f'No encuentro {PROYECTO_ZIP}'
assert os.path.exists(DATASET_ZIP),  f'No encuentro {DATASET_ZIP}'
print('Archivos encontrados')

In [ ]:
# 4. Descomprimir el proyecto y ubicarse en su raíz (funciona con o sin carpeta contenedora en el zip)
import glob, os
!rm -rf /content/_proyecto
!unzip -q "$PROYECTO_ZIP" -d /content/_proyecto
candidatos = glob.glob('/content/_proyecto/**/src/train_transfer.py', recursive=True)
assert candidatos, 'El zip del proyecto no contiene src/train_transfer.py'
PROJ = os.path.dirname(os.path.dirname(candidatos[0]))
%cd $PROJ
!ls

In [ ]:
# 5. Descomprimir el dataset en data/raw/{Positive,Negative}
import glob, os, shutil
!rm -rf data/raw && mkdir -p data/raw
!unzip -q "$DATASET_ZIP" -d data/raw
# Si el zip traía una carpeta contenedora, subir Positive/ y Negative/ al nivel data/raw
for clase in ('Positive', 'Negative'):
    if not os.path.isdir(f'data/raw/{clase}'):
        hallado = [d for d in glob.glob(f'data/raw/**/{clase}', recursive=True) if os.path.isdir(d)]
        assert hallado, f'No se encontró la carpeta {clase} dentro del dataset'
        shutil.move(hallado[0], f'data/raw/{clase}')
for clase in ('Positive', 'Negative'):
    n = len(glob.glob(f'data/raw/{clase}/*.jpg'))
    print(f'{clase}: {n} imágenes')

In [ ]:
# 6. Dependencias (Colab ya trae TensorFlow, OpenCV y scikit-learn; esto solo instala lo que falte)
import importlib
faltan = [m for m in ['tensorflow', 'cv2', 'sklearn', 'pandas', 'matplotlib'] if importlib.util.find_spec(m) is None]
if faltan:
    !pip -q install tensorflow opencv-python-headless scikit-learn pandas matplotlib
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 7. Análisis exploratorio rápido
!python src/dataset.py --root data/raw --max-por-clase 4000

In [ ]:
# 8. Entrenar MobileNetV2 (fase 1: cabeza, fase 2: fine-tuning) y exportar a TFLite
#    ~8 000 imágenes, 5 + 3 épocas. En T4 tarda unos 8-12 minutos.
!python src/train_transfer.py --root data/raw --max-por-clase 4000 --epochs 5 --epochs-finetune 3
!ls -la models/

In [ ]:
# 9. Verificación: el .tflite debe dar casi la misma probabilidad que el .keras
import numpy as np, tensorflow as tf, glob, cv2
from tensorflow import keras
m = keras.models.load_model('models/mobilenet.keras')
it = tf.lite.Interpreter(model_path='models/mobilenet.tflite'); it.allocate_tensors()
inp, out = it.get_input_details()[0], it.get_output_details()[0]
fotos = sorted(glob.glob('data/raw/Positive/*.jpg'))[:5] + sorted(glob.glob('data/raw/Negative/*.jpg'))[:5]
print(f"{'clase':10s} {'keras':>7s} {'tflite':>7s}")
for f in fotos:
    img = cv2.cvtColor(cv2.resize(cv2.imread(f), (224, 224)), cv2.COLOR_BGR2RGB).astype(np.float32)[None]
    pk = float(m.predict(img, verbose=0).ravel()[0])
    it.set_tensor(inp['index'], img); it.invoke()
    pt = float(it.get_tensor(out['index']).ravel()[0])
    print(f"{f.split('/')[-2]:10s} {pk:7.3f} {pt:7.3f}")
print('\nPositive debe dar valores cercanos a 1 y Negative cercanos a 0; keras y tflite deben coincidir (±0.02).')

In [ ]:
# 10. Curvas de entrenamiento
from IPython.display import Image, display
display(Image('models/curvas.png'))

In [ ]:
# 11. Guardar los modelos y métricas en Drive para bajarlos al PC
import os, shutil, glob
os.makedirs(SALIDA_DRIVE, exist_ok=True)
for f in glob.glob('models/mobilenet*') + ['models/curvas.png']:
    shutil.copy(f, SALIDA_DRIVE)
print('Copiado a', SALIDA_DRIVE)
!ls -la "$SALIDA_DRIVE"

## Siguiente paso en el PC

Descarga de Drive `mobilenet.tflite` y `mobilenet_metricas.json` a la carpeta `models/` del proyecto y usa la configuración
**9. Predecir con MobileNet** o **10. App web con MobileNet** en VS Code.